# yolo (OPTIONAL): an alternative piece detector

**Optional, not part of the M1-M5 deliverables.** Instead of classifying 64 fixed crops, a
detector (Ultralytics YOLO) finds every piece as a box + class in the warped image. Possible
advantages: it also gives the piece position (useful for `placement.ipynb`) and copes with
pieces that are off their square. Cost: box labels, a heavier model, another dependency.

Labels come for free from M2 capture: the manifest says which piece is on which square, and
the crop box of that square (`capture.ipynb: crop_box`) is a usable approximate bounding box.
Dataset config: `configs/yolo_data.yaml` at the repo root (12 classes = `CLASSES[1:]`).

**TODO** (only if you pursue this)
- [ ] `label_to_yolo_class`, `yolo_box`, `yolo_lines`, `write_yolo_sample`
- [ ] tighter boxes (piece height from `board.yaml: pieces.height`, perspective lean)
- [ ] `train_yolo` on Colab (`pip install ultralytics`), compare with the ResNet-18 crops
      on the same test sessions (convert detections to 64 labels via `placement.ipynb: which_square`)
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from pathlib import Path

import chess
import numpy as np
import yaml

%run ./capture.ipynb

In [ ]:
YOLO_NAMES = CLASSES[1:]                     # 12 piece classes
YOLO_DATA_YAML = REPO_ROOT / "configs" / "yolo_data.yaml"


def label_to_yolo_class(label: str) -> int:
    """YOLO class id = CLASSES index - 1 ("empty" has no box -> ValueError)."""
    # TODO: implement
    raise NotImplementedError

## Boxes from a known board
YOLO label line: `class x_center y_center width height`, all normalized to [0, 1].

In [ ]:
def yolo_box(box: tuple[int, int, int, int], img_w: int, img_h: int) -> tuple[float, float, float, float]:
    """Pixel box (x0, y0, x1, y1), clipped to the image -> normalized (xc, yc, w, h)."""
    # Design: clip to the image first (crop boxes may extend past the warp), then normalize.
    # TODO: implement
    raise NotImplementedError


def yolo_lines(board: chess.Board, geom: BoardGeometry, warp, crop, img_w: int, img_h: int,
               widen: float = 0.1) -> list[str]:
    """One label line per piece. Box = the square's crop box with a narrower `widen`."""
    # Design: copy the crop config with widen replaced (the crop's 0.5 is far too wide for a
    # box), crop_box per occupied square, yolo_box, format "cls xc yc w h" with 6 decimals.
    # TODO: implement
    raise NotImplementedError


def write_yolo_sample(root: str | Path, split: str, name: str, warped: np.ndarray,
                      board: chess.Board, geom: BoardGeometry, warp, crop) -> None:
    """Write images/<split>/<name>.png and labels/<split>/<name>.txt under root (Ultralytics layout)."""
    # TODO: implement
    raise NotImplementedError

## Training

In [ ]:
def train_yolo(data_yaml: str | Path = YOLO_DATA_YAML, model: str = "yolov8n.pt",
               epochs: int = 50, imgsz: int = 1024):
    """(optional) Fine-tune a YOLO model on the exported dataset."""
    # Design:
    # - ultralytics.YOLO(model).train(data=str(data_yaml), epochs=epochs, imgsz=imgsz).
    # - Decide the image size (the default warp is 1000 px) and the augmentations.
    # - Turn detections into a (64, 13) array for board_assembly.ipynb (box base point ->
    #   placement.ipynb: which_square, class score -> probability row).
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
_y = yaml.safe_load(YOLO_DATA_YAML.read_text())   # config consistency only, no stubs called
assert [_y["names"][i] for i in range(12)] == list(YOLO_NAMES)

# Self-check plan (write these as asserts once the functions above work):
# - label_to_yolo_class("P") == 0, ("k") == 11, ("empty") raises ValueError.
# - yolo_lines(start position, default camera.yaml warp + crop, 1000, 1000) -> 32 lines,
#   all values in [0, 1].
# - write_yolo_sample into a tempdir -> labels/train/000000.txt has 32 lines (+ trailing newline).
print("yolo: self-check not written yet (see plan above)")